In [6]:
from gam_rs_utils.utils import *
from src.prepare_gam import get_fastsparse
from src.rset_opt import RSetOPT
from src.run_app import get_models_from_rset
from method_scripts.results import Results

In [7]:
dn = 'bank'
data = pd.read_csv(f'datasets/{dn}.csv')
l0 = 0.001
l2 = 0.001
m = 1.01

# data is 17 features
# w becomes 3719 features

# feature: house
# 3  7 12
# 0  0  1 0 0
# 0  0  0 0 1

# 3,4,5,6,7
# 1 1 1 0 0
# 1 1 1 1 1

fastsparse_data = Results.create_fastsparse_dataset(dn, l0, l2)
bin_X = fastsparse_data['X']
y = fastsparse_data['y']
header = fastsparse_data['header']
sample_p = fastsparse_data['sample_proportion']
w = fastsparse_data['w']

print("data shape", data.shape)
print("w", len(w), "y", len(y), "header", len(header))
print("bin_X shape", bin_X.shape)

Loading cached fastsparse dataset for bank with l0=0.001 and l2=0.001
data shape (4521, 17)
w 3719 y 4521 header 3719
bin_X shape (4521, 3719)


In [8]:
# start = time()

sparse_X, sparse_header = utils.binary_to_one_hot(data.iloc[:,:-1], w, header)
# print("sparse_X shape", sparse_X.shape, "sparse_header", sparse_header)
sparse_gam_file = prepare_sparse_gam(dn, l0, l2, m, sparse_X, y, header, sparse_header)

model = RSetOPT(sparse_gam_file)
model.finetune_ellipsoid()
H_opt = model.get_normalized_H()
model.update_file(H_opt, model.w_orig)

# end = time()

with open(sparse_gam_file, 'rb') as f:
    sparse_gam_data = pickle.load(f)

objective: 0.2443484497913396 objective in LR 0.2443484497913396
m:1.01, log objective:0.2443484497913396, eps:0.246791934289253
----------- before optimization -----------
volume proportional to  tensor(3.5345e+24, dtype=torch.float64, grad_fn=<MulBackward0>)
----------- after optimization -----------
volume proportional to  tensor(1.4693e+24, dtype=torch.float64, grad_fn=<MulBackward0>)


In [9]:
n_samples = 100
sampling = 'uniform'
distance_metric = None
r_min = None

w_samples, rset = get_models_from_rset(
    sparse_gam_file, n_samples=n_samples, plot_shape=False, 
    sampling=sampling, distance_metric=distance_metric, r_min=r_min,
)

print('w_samples shape', w_samples.shape)

w_samples shape (100, 21)


In [5]:
# w_samples_zeroed = ModelUtils.hard_threshold_samples(w_samples, rset, 17)
# ModelUtils.print_results_summary(w_samples_zeroed, sparse_gam_data['w_opt'], X, y, l2, sample_p, end - start)

array([-2.04192107, -2.44651175, -3.86067603, ..., -3.22057456,
       -2.72400194, -2.44651175])

In [10]:
header_object = ModelUtils.get_header_object(header)
sparse_header_object = ModelUtils.get_header_object(sparse_header)

expanded_w_samples = ModelUtils.expand_w_samples(w_samples, sparse_header_object, header_object)

In [11]:
for i in range(len(w_samples)):
    expanded_w = expanded_w_samples[i]
    w = w_samples[i]
    a = ModelUtils.get_logits(sparse_X, w)
    b = ModelUtils.get_logits(bin_X, expanded_w)
    if not np.allclose(a,b):
        print(f'w_samples[{i}] is not equal to expanded_w')
        print(a - b)

In [14]:
expanded_sample_p = bin_X.sum(0) / bin_X.shape[0]
loss_a, _ = ModelUtils.get_loss(bin_X, y, expanded_w_samples, loss_type='logistic', l2=0.001, sample_p=expanded_sample_p)
loss_b, _ = ModelUtils.get_loss(sparse_X, y, w_samples, loss_type='logistic', l2=0.001, sample_p=sample_p)
np.allclose(loss_a, loss_b)

True